# Laboratorio 4, Parte 2 - Analisis exploratorio geoespacial

**Curso:** CC3084 Ciencia de Datos, Universidad del Valle de Guatemala  
**Unidad de analisis:** pixel Sentinel-2 de 20 m por lago y fecha.

Este cuaderno visualiza distribuciones espectrales, desbalance de la respuesta y ubicacion espacial. La carga se realiza por bloques para no agotar memoria y la muestra de graficacion es determinista. Los conteos de clases siempre usan todas las observaciones.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

raiz_laboratorio = Path.cwd()
if raiz_laboratorio.name == 'notebooks':
    raiz_laboratorio = raiz_laboratorio.parent
if not (raiz_laboratorio / 'data').is_dir():
    raise FileNotFoundError('Ejecute el notebook desde Lab4.2 o Lab4.2/notebooks.')

ruta_datos = raiz_laboratorio / 'data' / 'processed' / '02_datos_con_respuesta.csv'
directorio_resultados = raiz_laboratorio / 'results'
directorio_resultados.mkdir(parents=True, exist_ok=True)
if not ruta_datos.is_file():
    raise FileNotFoundError('Ejecute primero los scripts 01 y 02 del pipeline.')

sns.set_theme(style='whitegrid', context='notebook')
plt.rcParams.update({'figure.dpi': 120, 'savefig.dpi': 180})
semilla = 3084
print(f'Datos: {ruta_datos}')

In [ ]:
def cargar_eda_por_bloques(ruta, tamano_bloque=200_000, muestra_por_bloque=10_000):
    """Carga una muestra reproducible y agrega conteos sobre el conjunto completo."""
    columnas = [
        'lago', 'fecha', 'coordenada_x', 'coordenada_y',
        'b02', 'b03', 'b04', 'b05', 'b08', 'ndvi', 'ndwi',
        'presencia_alta_cianobacterias',
    ]
    muestras = []
    conteos = []
    total = 0
    for numero_bloque, bloque in enumerate(
        pd.read_csv(ruta, usecols=columnas, chunksize=tamano_bloque), start=1
    ):
        if bloque[columnas].isna().any().any():
            raise ValueError(f'Valores faltantes en el bloque {numero_bloque}.')
        cantidad = min(muestra_por_bloque, len(bloque))
        muestras.append(bloque.sample(n=cantidad, random_state=semilla + numero_bloque))
        conteos.append(
            bloque.groupby(
                ['lago', 'fecha', 'presencia_alta_cianobacterias'], observed=True
            ).size().rename('observaciones').reset_index()
        )
        total += len(bloque)
    if total == 0:
        raise ValueError('El archivo procesado no contiene observaciones.')
    muestra = pd.concat(muestras, ignore_index=True)
    distribucion = (
        pd.concat(conteos, ignore_index=True)
        .groupby(['lago', 'fecha', 'presencia_alta_cianobacterias'], as_index=False)[
            'observaciones'
        ].sum()
    )
    return muestra, distribucion, total

datos_muestra, distribucion_clases, total_observaciones = cargar_eda_por_bloques(ruta_datos)
print(f'Observaciones completas: {total_observaciones:,}')
print(f'Muestra determinista para distribuciones y mapas: {len(datos_muestra):,}')
display(datos_muestra.head())

## Distribuciones espectrales por lago

Los histogramas permiten observar asimetria y multimodalidad; los boxplots facilitan comparar mediana, dispersion y valores extremos. B04 y B05 se muestran solo con fines exploratorios: no pertenecen al conjunto final de predictores porque construyen la etiqueta.

In [ ]:
variables_distribucion = ['ndvi', 'ndwi', 'b03', 'b04', 'b05', 'b08']
figura, ejes = plt.subplots(2, 3, figsize=(15, 8), constrained_layout=True)
for eje, variable in zip(ejes.flat, variables_distribucion):
    sns.histplot(
        data=datos_muestra, x=variable, hue='lago', bins=45,
        stat='density', common_norm=False, element='step', fill=False, ax=eje,
    )
    eje.set_title(f'Distribucion de {variable.upper()}')
    eje.set_ylabel('Densidad')
figura.suptitle('Distribuciones espectrales por lago', fontsize=16)
figura.savefig(directorio_resultados / '01_histogramas_por_lago.png', bbox_inches='tight')
plt.show()

In [ ]:
datos_largos = datos_muestra.melt(
    id_vars='lago', value_vars=variables_distribucion,
    var_name='variable', value_name='valor',
)
figura, ejes = plt.subplots(2, 3, figsize=(15, 8), constrained_layout=True)
for eje, variable in zip(ejes.flat, variables_distribucion):
    subconjunto = datos_largos[datos_largos['variable'] == variable]
    sns.boxplot(
        data=subconjunto, x='lago', y='valor', hue='lago',
        showfliers=False, legend=False, ax=eje,
    )
    eje.set_title(variable.upper())
    eje.set_xlabel('')
figura.suptitle('Comparacion robusta por lago (sin dibujar atipicos)', fontsize=16)
figura.savefig(directorio_resultados / '02_boxplots_por_lago.png', bbox_inches='tight')
plt.show()

## Distribucion y desbalance de la variable respuesta

Los siguientes conteos se calculan con el conjunto completo, no con la muestra usada para graficar variables continuas. La clase 0 representa ausencia o baja presencia y la clase 1 alta presencia operativa (indice mayor o igual que 20 mg/m3).

In [ ]:
distribucion_global = (
    distribucion_clases.groupby('presencia_alta_cianobacterias', as_index=False)[
        'observaciones'
    ].sum()
)
distribucion_global['porcentaje'] = (
    100 * distribucion_global['observaciones'] / distribucion_global['observaciones'].sum()
)
distribucion_lago = (
    distribucion_clases.groupby(
        ['lago', 'presencia_alta_cianobacterias'], as_index=False
    )['observaciones'].sum()
)
distribucion_lago['porcentaje'] = distribucion_lago.groupby('lago')[
    'observaciones'
].transform(lambda valores: 100 * valores / valores.sum())

figura, ejes = plt.subplots(1, 2, figsize=(13, 5), constrained_layout=True)
sns.barplot(
    data=distribucion_global, x='presencia_alta_cianobacterias', y='porcentaje',
    hue='presencia_alta_cianobacterias', legend=False, ax=ejes[0],
)
ejes[0].set(title='Distribucion global', xlabel='Clase', ylabel='Porcentaje')
sns.barplot(
    data=distribucion_lago, x='lago', y='porcentaje',
    hue='presencia_alta_cianobacterias', ax=ejes[1],
)
ejes[1].set(title='Distribucion por lago', xlabel='', ylabel='Porcentaje')
ejes[1].legend(title='Clase')
figura.savefig(directorio_resultados / '03_clases_global_lago.png', bbox_inches='tight')
plt.show()
display(distribucion_global)
display(distribucion_lago)

In [ ]:
distribucion_fecha = distribucion_clases.copy()
distribucion_fecha['porcentaje'] = distribucion_fecha.groupby(['lago', 'fecha'])[
    'observaciones'
].transform(lambda valores: 100 * valores / valores.sum())
grafico = sns.catplot(
    data=distribucion_fecha, kind='bar', x='fecha', y='porcentaje',
    hue='presencia_alta_cianobacterias', row='lago', height=4, aspect=2.5,
    sharex=False, sharey=True,
)
grafico.set_axis_labels('Fecha', 'Porcentaje de pixeles')
grafico.set_titles('{row_name}')
grafico.set_xticklabels(rotation=45, ha='right')
grafico.figure.suptitle('Distribucion de clases por fecha y lago', y=1.02, fontsize=15)
grafico.figure.savefig(directorio_resultados / '04_clases_por_fecha.png', bbox_inches='tight')
plt.show()

In [ ]:
conteos_globales = distribucion_global.set_index('presencia_alta_cianobacterias')[
    'observaciones'
]
mayoritaria = int(conteos_globales.max())
minoritaria = int(conteos_globales.min())
razon_desbalance = mayoritaria / max(minoritaria, 1)
porcentaje_minoritaria = 100 * minoritaria / conteos_globales.sum()
print(f'Razon clase mayoritaria:minoritaria = {razon_desbalance:.2f}:1')
print(f'Participacion de la clase minoritaria = {porcentaje_minoritaria:.4f}%')
if porcentaje_minoritaria < 10:
    print('Diagnostico: desbalance severo; Accuracy no debe ser la metrica principal.')
elif porcentaje_minoritaria < 30:
    print('Diagnostico: desbalance moderado; reportar metricas sensibles a la clase 1.')
else:
    print('Diagnostico: no se observa un desbalance fuerte en el agregado global.')

## Mapa exploratorio de la respuesta

La figura conserva las coordenadas proyectadas EPSG:32615. Es un mapa de puntos de una muestra determinista, no un mapa de riesgo ni una interpolacion.

In [ ]:
figura, ejes = plt.subplots(1, 2, figsize=(14, 6), constrained_layout=True)
for eje, (lago, datos_lago) in zip(ejes, datos_muestra.groupby('lago')):
    eje.scatter(
        datos_lago['coordenada_x'], datos_lago['coordenada_y'],
        c=datos_lago['presencia_alta_cianobacterias'], cmap='coolwarm',
        vmin=0, vmax=1, s=2, alpha=0.35, rasterized=True,
    )
    eje.set(title=lago, xlabel='Este (m)', ylabel='Norte (m)', aspect='equal')
figura.suptitle('Distribucion espacial exploratoria de la respuesta', fontsize=16)
figura.savefig(directorio_resultados / '05_mapa_respuesta.png', bbox_inches='tight')
plt.show()

## Criterios para la etapa de modelado

1. Reportar matriz de confusion, precision, recall/sensibilidad, F1 para la clase 1 y PR-AUC; ROC-AUC puede incluirse solo como complemento.
2. Comparar contra un clasificador base que siempre prediga la clase mayoritaria.
3. Separar entrenamiento y prueba por fecha o por bloques espaciales. Una particion aleatoria por pixel filtraria autocorrelacion espacial entre conjuntos.
4. Ajustar pesos de clase o remuestreo solo dentro de cada pliegue de entrenamiento. Nunca balancear antes de la particion.
5. NDVI, NDCI, B04 y B05 son validos para EDA, pero no para entrenar esta etiqueta determinista.